In [ ]:
# Question 1 — Reading Payoff Matrices

matrix = [
    [(3, 3), (0, 5)],   # P1 plays Top
    [(5, 0), (1, 1)]    # P1 plays Bottom
]

# (a) P1 plays Top, P2 plays Right → matrix[0][1] → P1 gets 0
# (b) Both play Bottom → matrix[1][1] → P2 gets 1
# (c) When P2 plays Left (col 0): Top gives P1 = 3, Bottom gives P1 = 5 → Bottom is better
#     When P2 plays Right (col 1): Top gives P1 = 0, Bottom gives P1 = 1 → Bottom is better

In [ ]:
# Question 2 — Dominant Strategy Finder (2-Player)

def find_dominant_strategy(matrix):
    rows = len(matrix)
    cols = len(matrix[0])

    # Check P1 rows for strict dominance
    p1_dominant = None
    for r in range(rows):
        is_dominant = True
        for other_r in range(rows):
            if other_r == r:
                continue
            # r must beat other_r in every column
            for c in range(cols):
                if matrix[r][c][0] <= matrix[other_r][c][0]:
                    is_dominant = False
                    break
            if not is_dominant:
                break
        if is_dominant:
            p1_dominant = r
            break

    # Check P2 columns for strict dominance
    p2_dominant = None
    for c in range(cols):
        is_dominant = True
        for other_c in range(cols):
            if other_c == c:
                continue
            for r in range(rows):
                if matrix[r][c][1] <= matrix[r][other_c][1]:
                    is_dominant = False
                    break
            if not is_dominant:
                break
        if is_dominant:
            p2_dominant = c
            break

    if p1_dominant is not None:
        print(f"Player 1 has a strictly dominant strategy: Row {p1_dominant}")
    else:
        print("Player 1 has no strictly dominant strategy.")

    if p2_dominant is not None:
        print(f"Player 2 has a strictly dominant strategy: Column {p2_dominant}")
    else:
        print("Player 2 has no strictly dominant strategy.")


# (i) Advertise / Not-Advertise
advertise_matrix = [
    [(6, 4), (8, 1)],
    [(4, 6), (7, 3)]
]
print("--- Advertise / Not-Advertise ---")
find_dominant_strategy(advertise_matrix)

# (ii) Prisoner's Dilemma
prisoners_matrix = [
    [(-1, -1), (-3,  0)],
    [( 0, -3), (-2, -2)]
]
print("\n--- Prisoner's Dilemma ---")
find_dominant_strategy(prisoners_matrix)

# (iii) Stag Hunt
stag_hunt_matrix = [
    [(4, 4), (0, 2)],
    [(2, 0), (2, 2)]
]
print("\n--- Stag Hunt ---")
find_dominant_strategy(stag_hunt_matrix)

In [ ]:
# Question 3 — IESDS Solver (2-Player)

def run_iesds(matrix):
    # Track which rows/cols are still alive
    live_rows = list(range(len(matrix)))
    live_cols = list(range(len(matrix[0])))

    changed = True
    while changed:
        changed = False

        # Check if any row of P1 is strictly dominated
        row_to_remove = None
        dominator_row = None
        for r in live_rows:
            for other_r in live_rows:
                if other_r == r:
                    continue
                # other_r strictly dominates r if it beats r in every live column
                if all(matrix[other_r][c][0] > matrix[r][c][0] for c in live_cols):
                    row_to_remove = r
                    dominator_row = other_r
                    break
            if row_to_remove is not None:
                break

        if row_to_remove is not None:
            print(f"Removing Row {row_to_remove}: strictly dominated by Row {dominator_row}")
            live_rows.remove(row_to_remove)
            changed = True
            continue

        # Check if any column of P2 is strictly dominated
        col_to_remove = None
        dominator_col = None
        for c in live_cols:
            for other_c in live_cols:
                if other_c == c:
                    continue
                if all(matrix[r][other_c][1] > matrix[r][c][1] for r in live_rows):
                    col_to_remove = c
                    dominator_col = other_c
                    break
            if col_to_remove is not None:
                break

        if col_to_remove is not None:
            print(f"Removing Col {col_to_remove}: strictly dominated by Col {dominator_col}")
            live_cols.remove(col_to_remove)
            changed = True

    print(f"\nSurviving rows: {live_rows}")
    print(f"Surviving cols: {live_cols}")

    if len(live_rows) == 1 and len(live_cols) == 1:
        r, c = live_rows[0], live_cols[0]
        print(f"Predicted outcome: Row {r}, Col {c} → payoffs {matrix[r][c]}")
    else:
        print("No unique outcome from IESDS.")


# 3x4 matrix from slides (rows A/B/C, cols D/E/F/G)
# Should eliminate down to (B, D)
slides_matrix = [
    [(1,1), (-1,2), (5,0), (1,1)],   # Row A
    [(2,3), ( 1,2), (3,0), (5,1)],   # Row B
    [(1,1), ( 0,5), (1,7), (0,1)]    # Row C
]
print("--- 3x4 IESDS from slides (A/B/C vs D/E/F/G) ---")
run_iesds(slides_matrix)

In [ ]:
# Question 4 — Nash Equilibrium Finder: Pure Strategies (2-Player)

def find_pure_nash(matrix, name=""):
    rows = len(matrix)
    cols = len(matrix[0])

    # Best responses for P1: for each column, find rows where P1 payoff is max
    p1_best = set()
    for c in range(cols):
        best_payoff = max(matrix[r][c][0] for r in range(rows))
        for r in range(rows):
            if matrix[r][c][0] == best_payoff:
                p1_best.add((r, c))

    # Best responses for P2: for each row, find cols where P2 payoff is max
    p2_best = set()
    for r in range(rows):
        best_payoff = max(matrix[r][c][1] for c in range(cols))
        for c in range(cols):
            if matrix[r][c][1] == best_payoff:
                p2_best.add((r, c))

    nash_cells = p1_best & p2_best

    if name:
        print(f"--- {name} ---")
    if nash_cells:
        for r, c in sorted(nash_cells):
            print(f"Nash Equilibrium at (Row {r}, Col {c}): payoffs {matrix[r][c]}")
    else:
        print("No pure-strategy Nash Equilibrium found.")
    print()


# (i) Prisoner's Dilemma
pd_matrix = [
    [(-1, -1), (-3,  0)],
    [( 0, -3), (-2, -2)]
]
find_pure_nash(pd_matrix, "Prisoner's Dilemma")

# (ii) Stag Hunt
stag_matrix = [
    [(4, 4), (0, 2)],
    [(2, 0), (2, 2)]
]
find_pure_nash(stag_matrix, "Stag Hunt")

# (iii) Chicken
chicken_matrix = [
    [( 0,  0), (-1,  1)],
    [( 1, -1), (-10, -10)]
]
find_pure_nash(chicken_matrix, "Chicken")

# (iv) Battle of the Sexes
bos_matrix = [
    [(3, 2), (0, 0)],
    [(0, 0), (2, 3)]
]
find_pure_nash(bos_matrix, "Battle of the Sexes")

# (v) 3x3 matrix from slides (A/B/C vs X/Y/Z)
slides_3x3 = [
    [(4, 4), (0, 0), (1, 3)],   # Row A
    [(0, 0), (5, 5), (0, 0)],   # Row B
    [(3, 1), (0, 0), (6, 6)]    # Row C
]
find_pure_nash(slides_3x3, "3x3 Slides Matrix (A/B/C vs X/Y/Z)")

In [ ]:
# Question 5 — Nash Equilibrium Finder: N-Player Game

import itertools

def find_nash_n_players(strategies, payoffs):
    all_profiles = list(itertools.product(*strategies))
    nash_list = []

    for profile in all_profiles:
        is_nash = True
        for player in range(len(strategies)):
            current_payoff = payoffs[profile][player]
            # Try every other strategy for this player
            for alt_strat in strategies[player]:
                if alt_strat == profile[player]:
                    continue
                # Build the deviated profile
                deviated = list(profile)
                deviated[player] = alt_strat
                deviated = tuple(deviated)
                if payoffs[deviated][player] > current_payoff:
                    is_nash = False
                    break
            if not is_nash:
                break
        if is_nash:
            nash_list.append(profile)

    if nash_list:
        print("Nash Equilibria found:")
        for ne in nash_list:
            print(f"  {ne} → payoffs {payoffs[ne]}")
    else:
        print("No pure-strategy Nash Equilibrium exists.")

    return nash_list


# 3-player Cooperate/Defect game
strategies = [['C', 'D'], ['C', 'D'], ['C', 'D']]
payoffs = {
    ('C','C','C'): [4, 4, 4],
    ('C','C','D'): [1, 1, 6],
    ('C','D','C'): [1, 6, 1],
    ('C','D','D'): [0, 3, 3],
    ('D','C','C'): [6, 1, 1],
    ('D','C','D'): [3, 0, 3],
    ('D','D','C'): [3, 3, 0],
    ('D','D','D'): [2, 2, 2],
}
print("--- 3-Player Cooperate/Defect ---")
find_nash_n_players(strategies, payoffs)

# The Nash Equilibrium is (D, D, D) with payoffs [2, 2, 2].
# This is not surprising — it mirrors the classic Prisoner's Dilemma logic scaled to three players.
# Each firm individually gains by defecting regardless of what the others do,
# so the only stable outcome is universal defection, even though (C,C,C) with payoffs [4,4,4]
# would be collectively better. It illustrates the tragedy of the commons:
# rational self-interest leads to a worse outcome for everyone.

In [ ]:
# Question 6 — Interactive Game Solver

def interactive_solver():
    print("=== Interactive Two-Player Game Solver ===")
    num_rows = int(input("How many strategies does Player 1 have (rows)? "))
    num_cols = int(input("How many strategies does Player 2 have (cols)? "))

    matrix = []
    print("\nEnter payoffs row by row.")
    print("For each cell, enter two space-separated integers: p1_payoff p2_payoff")
    for r in range(num_rows):
        row = []
        for c in range(num_cols):
            raw = input(f"  Row {r}, Col {c}: ")
            p1, p2 = map(int, raw.strip().split())
            row.append((p1, p2))
        matrix.append(row)

    # Display matrix
    print("\n--- Your Matrix ---")
    header = "        " + "  ".join(f"Col {c}" for c in range(num_cols))
    print(header)
    for r in range(num_rows):
        row_str = f"Row {r}:  " + "  ".join(str(matrix[r][c]) for c in range(num_cols))
        print(row_str)

    print("\n--- Dominant Strategy Analysis ---")
    find_dominant_strategy(matrix)

    print("\n--- IESDS ---")
    run_iesds(matrix)

    print("--- Nash Equilibria (Pure Strategies) ---")
    find_pure_nash(matrix)


# Test with Battle of the Sexes:
# rows = 2, cols = 2
# Row 0, Col 0: 3 2
# Row 0, Col 1: 0 0
# Row 1, Col 0: 0 0
# Row 1, Col 1: 2 3

interactive_solver()